# Scenario 4: Python class for reusable functions

In a large-scale data processing project, multiple PySpark notebooks require the same set of custom transformation functions, such as **date formatting**, **null handling**, and **data validation**. Instead of duplicating the code across notebooks, a Python class is created to store these reusable functions. This ensures consistency, reduces maintenance effort, and improves code readability across the project.

## Data Validation

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Validate the data before we upsert 
class DataValidation:

    def __init__(self, df):         # these columns will be filled by user
        self.df = df

    def dedup(self, keyCol, cdcCol):
        self.df = self.df.withColumn("dedup", row_number().over(Window
                                                      .partitionBy(keyCol)
                                                      .orderBy(desc(cdcCol))))
        self.df = self.df.filter(col("dedup") == 1).drop("dedup")

        return self.df

    def removeNulls(self, nulCol):
        self.df = self.df.filter(col(nulCol).isNotNull())
        
        return self.df

In [0]:
df = (spark.createDataFrame(
    [
        ("1", "2020-01-01", 100),
        ("1", "2020-01-01", 100),
        ("2", "2020-01-02", 200),
        ("3", "2020-01-01", 300),
    ],
    [
        "order_id", "order_date", "amount"
    ]
))
display(df)

order_id,order_date,amount
1,2020-01-01,100
1,2020-01-01,100
2,2020-01-02,200
3,2020-01-01,300


In [0]:
cls_obj = DataValidation(df)

In [0]:
df_dedup = cls_obj.dedup("order_id", "order_date")

display(df_dedup)

order_id,order_date,amount
1,2020-01-01,100
2,2020-01-02,200
3,2020-01-01,300
